## Welcome to my Sentinels Valorant Data Analysis Project

I'll be taking a look data on the Sentinels Valorant Team across 2 different regional events, looking at statistics from custom datasets to look at what the team has struggled with, improved on, and seeing if changes they made were worth it  
For some context, the team came into 2026 losing most of their roster and introducing a couple of rookies, as well as 2 crucial role swaps that seemed to have defined their Kickoff. After they were knocked out of the tournament, they fired their coach Kaplan and made 2 roster moves sending away Kyu and N4rrate switching in-game leaders back to their original johnqt. The 2 replacements for Kyu and N4rrate were JonahP and tier-2 upcoming star Jerrwin.

First, we take a look at our datasets below

Dataset 1: all_players_kickoff.csv  
This dataset contains overall individual stats across all the players who participated in VCT Americas Kickoff: 2026  
This data was collected via a WebScrape from the website url "https://www.vlr.gg/event/stats/2860/vct-2026-americas-stage-1"

Dataset 2: all_players_s1.csv  
This dataset contains overall individual statistics across all the players who participated in VCT Americas Stage 1: 2026  
This data was also collected via a WebScrape from the website url "https://www.vlr.gg/event/stats/2860/vct-2026-americas-stage-1"

Dataset 3: individual_sen_series_stats.csv  
This dataset contains individual statistics across different series for each of the Sentinels players across both Kickoff and Stage 1  
This data was collected via manual inputs from each series Sentinels participated in (9 series)

Dataset 4: regional_team_stats_kickoff.csv  
This dataset contains overall regional team statistics pertaining to round results / winrates  
This data was collected via manual inputs from "https://www.thespike.gg/valorant-stats/teams#event=3944"

Dataset 5: regional_team_stats_s1.csv  
This dataset is identical to Dataset 4, however is from "https://www.thespike.gg/valorant-stats/teams#event=4044"

Dataset 6: vod_review.csv  
This dataset contains round-by-round data collection by each individual on the Sentinels team, tracking every round, every map, and every series from Kickoff and Stage 1

Before we work with the datasets, we have to clean them, but before we clean them, we have to make some imports before we can have functional code!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import requests # For scraping
from bs4 import BeautifulSoup # For scraping

Dataset 1 + 2 Scraping

In [ ]:
def scrape_vlr(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("table", class_="wf-table mod-stats mod-scroll")

    if table is None:
        raise ValueError("Could not find stats table.")

    rows = []

    for tr in table.find("tbody").find_all("tr"):
        tds = tr.find_all("td")

        if len(tds) < 14:
            continue

        row = {
            "player": tds[0].get_text(strip=True),
            "rating": tds[3].get_text(strip=True),
            "acs": tds[4].get_text(strip=True),
            "kd": tds[5].get_text(strip=True),
            "kast": tds[6].get_text(strip=True),
            "adr": tds[7].get_text(strip=True),
            "kpr": tds[8].get_text(strip=True),
            "apr": tds[9].get_text(strip=True),
            "fkpr": tds[10].get_text(strip=True),
            "fdpr": tds[11].get_text(strip=True),
            "hs_percent": tds[12].get_text(strip=True),
            "cl_percent": tds[13].get_text(strip=True),
        }

        rows.append(row)

    df = pd.DataFrame(rows)

    return df

all_players_kickoff_df = scrape_vlr("https://www.vlr.gg/event/stats/2682/vct-2026-americas-kickoff")
all_players_s1_df = scrape_vlr("https://www.vlr.gg/event/stats/2860/vct-2026-americas-stage-1")

all_players_kickoff_df.head()
all_players_s1_df.head()

Dataset 3 Cleaning

In [ ]:
def sen_series_clean(input_path):
    df = pd.read_csv(input_path)
    
    for col in ["kast", "hs_percent", "cl_percent"]:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.replace("%", "", regex=False)
                .astype(float) / 100
            )

    df['cl%'] = df['cl%'].fillna(0)

    numeric_cols = [
        "series_result", "rds_played", "rating", "acs", "kd", "kast", "adr", "kpr", "apr", "fkpr", "fdpr", "hs_percent", "cl_percent"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

sen_indiv_clean_df = sen_series_clean("../raw_data/individual_sen_series_stats.csv")

sen_indiv_clean_df.head()

Dataset 4 + 5 Cleaning

In [ ]:
def teams_clean(input_path):
    df = pd.read_csv(input_path)

    numeric_cols = [
        "wr", "t_wr", "ct_wr", "p_wr"
    ]

    for col in numeric_cols:
        if col in df.columns:
            frac = df[col].str.split("/", expand=True).astype(float)
            df[col] = frac[0] / frac[1] * 100

    return df

teams_kickoff_df = teams_clean("../raw_data/reg_teams_kickoff.csv")
teams_s1_df = teams_clean("../raw_data/reg_teams_s1.csv")

teams_kickoff_df.head()
teams_s1_df.head()

Dataset 6 Cleaning

In [ ]:
def vod_clean(input_path):
    df = pd.read_csv(input_path)

    numeric_cols = [
        "round", "spike_planted", "plant_time", "fk_time", "round_win", "johnqt_k_pre", "reduxx_k_pre", "cortezia_k_pre", "n4rrate_k_pre", "kyu_k_pre",
        "jonahp_k_pre", "victor_k_pre", "jerrwin_k_pre", "johnqt_k_post", "reduxx_k_post", "cortezia_k_post", "n4rrate_k_post", "kyu_k_post", "jonahp_k_post", 
        "victor_k_post", "jerrwin_k_post"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

vod_df = vod_clean("../raw_data/vod_review.csv")

vod_df.head()